# S6E9 · Darwin Cage on the residual: 47,679 programs, zero discoveries, and a power analysis

**The question.** After an honest model, is there *any* structure left that we have not thought of?

**The approach (Darwin Cage).** Do not tell the search what to look for or how. Start from random programs built out
of the thirteen raw columns (remainders, integer divisions, decimal digits, bit shifts, categorical values, up to four
atoms joined into one key), mutate and recombine them, and reward each program **only** by how well its groups predict
the residual of the best blend, measured out of fold. One fold is never touched by the search: a program counts as a
discovery only if it survives there.

**The result.** On the real residual: nothing survives. On synthetic labels with a planted structure of known size:
the same search finds it. That second part is what makes the first one mean something.

Repository with the package, tests and all logs: [https://github.com/Agnuxo1/s6e9-honest-ceiling](https://github.com/Agnuxo1/s6e9-honest-ceiling) (Apache-2.0).

## 1 · Setup

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def find(name, roots=("/kaggle/input", os.environ.get("HC_LOCAL", "."))):
    """First file called `name` under the given roots (handles legacy and namespaced Kaggle input paths)."""
    for r in roots:
        hits = sorted(glob.glob(f"{r}/**/{name}", recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(name)

try:
    import honest_ceiling  # noqa: F401
except ImportError:  # the wheel ships inside the companion dataset; no internet needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find("honest_ceiling-1.0.0-py3-none-any.whl"))], check=True)
import honest_ceiling
from honest_ceiling.data import load, outer_folds
from honest_ceiling.blend import rank01, fast_auc, nested_hillclimb, paired_bootstrap

COMP = find("train.csv").parent
LIB = find("oof_legs.csv").parent            # companion dataset: our OOF/test library, folds, results
print("honest_ceiling", honest_ceiling.__version__, "| competition data:", COMP, "| library:", LIB)
train, test, y = load(COMP)
fold = outer_folds(y)
print(f"train {train.shape}  test {test.shape}  positives {y.mean():.4f}")

plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#e6e5e1",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "axes.titleweight": "semibold", "axes.titlelocation": "left", "font.size": 10})
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#b9b8b2"

## 2 · The residual
`blend_v4_nested` is the nested out-of-fold score of the best honest blend (see the companion pipeline notebook).
Two-way cross-fitted isotonic calibration turns it into a probability p; the residual is r = y − p.

In [ ]:
from honest_ceiling.residual import calibrated_probability, whiteness_report, oracle_bound
lib = pd.read_csv(LIB / "oof_legs.csv")
p = calibrated_probability(lib["blend_v4_nested"].to_numpy(float), y)
r = y - p
print(f"nested blend AUC {fast_auc(y, lib['blend_v4_nested'].to_numpy()):.6f}   residual mean {r.mean():+.5f}   std {r.std():.4f}")

## 3 · Is the residual white?
If the residual were correlated between rows in some ordering (id, income, commute, predicted probability), a filter
or a feature could exploit it. The ±2σ band of white noise at this size is ±0.0024.

In [ ]:
inc = train["Annual_Income_USD"].round().astype(int).to_numpy()
rep = whiteness_report(r, {"id": train["id"].to_numpy(), "income": inc, "commute": train["Daily_Commute_km"].to_numpy(),
                           "prediction": p}, {"exact income": inc})
print(pd.DataFrame(rep["acf"]).round(4).to_string())
print("\nleave-one-out residual correlation among rows sharing an exact income:", round(rep["group_loo_corr"]["exact income"], 4))

Everything sits inside or near the white-noise band. The small negative values in income order, and the negative
same-income correlation, have the sign cross-fitting produces (a row's label enters its same-income neighbours'
encodings in other folds): the income memory has been extracted, not left behind.

## 4 · The oracle bound
If p were the *true* probability of every row, the best achievable AUC is the AUC of p against labels drawn from p.

In [ ]:
ob = oracle_bound(p, reps=10)
print(f"oracle bound {ob['mean']:.5f} ± {ob['std']:.5f}   vs achieved {fast_auc(y, lib['blend_v4_nested'].to_numpy()):.5f}")

## 5 · The Darwin-Cage search, live
Fitness: shrunk group mean of r fit on 8 search folds, correlated with r on the 9th, averaged over the 9 rotations.
Fold 9 is reserved. Threshold for a discovery: 3σ of the null on that fold, `3 / sqrt(66,866) = 0.0116`.
This cell runs a short search (set `MINUTES`); the full runs (47,679 programs) are loaded in section 6.

In [ ]:
from honest_ceiling.darwin import DarwinSearch, integer_columns, plant, program_key
MINUTES = 15
D = integer_columns(train)
search = DarwinSearch(D, r.astype(np.float64), fold)
null_mean, null_std = search.null(reps=10)
print(f"null fitness (random keys): {null_mean:+.5f} ± {null_std:.5f}   threshold on the reserved fold: {search.threshold:.4f}")
res = search.run(seconds=MINUTES * 60, seed=11)
print(f"evaluated {res['evaluated']:,} programs in {res['seconds']:.0f} s | hall of fame {res['n_hall']} | confirmed {len(res['confirmed'])}"
      f" | expected family-wise false positives {res['familywise_fp_expected']:.3f}")
for h in res["hall"][:8]:
    print(f"  search {h['fit_mean']:.4f}  reserved fold {h['confirm']:+.4f}  {h['program']}")

## 6 · The full runs

In [ ]:
hall = []
for f in ("darwin_run1_hall.json", "darwin_run2_hall.json"):
    j = json.load(open(LIB / f))
    hall += [(h["fit_mean"], h.get("confirm_fold9", h.get("confirm")), h["program"]) for h in j["hall"]]
    print(f, j["report"]["evaluated"], "programs,", j["report"]["hall"], "hall entries,", j["report"]["confirmed"], "confirmed")
fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0.012, color=ORANGE, ls="--", lw=1.5); ax.text(0.0061, 0.0126, "discovery threshold (3σ, reserved fold)", fontsize=9, color="#52514e")
ax.axhspan(-0.008, 0.008, color=GREY, alpha=0.25, lw=0)
ax.scatter([h[0] for h in hall], [h[1] for h in hall], s=46, color=BLUE, edgecolor="white", zorder=3)
ax.set_xlabel("search fitness (9 search folds)"); ax.set_ylabel("correlation on the never-seen fold")
ax.set_title("Real residual: nothing that looks strong on the search folds survives the reserved fold")
plt.show()

## 7 · Power: would the search have found something?
Labels are redrawn from the real probabilities plus a planted per-group shift of known size d on a program the search
does not know about. Same search, same reserved fold. Below, one planted structure live, then the full power table.

In [ ]:
prog = [("mod", "Annual_Income_USD", 97)]
y_pl, shift = plant(p, D, prog, effect=0.016, seed=7)
gain = fast_auc(y_pl, p + shift) - fast_auc(y_pl, p)
s2 = DarwinSearch(D, (y_pl - p).astype(np.float64), fold)
t0 = time.time()
found = s2.run(seconds=15 * 60, seed=107, stop_on_confirm=True)
print(f"planted income mod 97, d = 0.016 (oracle AUC gain {gain:.5f}) -> confirmed {len(found['confirmed'])} after "
      f"{found['evaluated']:,} programs, {time.time()-t0:.0f} s")
for h in found["confirmed"][:3]:
    print("  ", h["program"], "reserved-fold correlation", round(h["confirm"], 4))

In [ ]:
rows = [json.load(open(f)) for f in sorted(glob.glob(str(LIB / "power_*.json")))]
tab = pd.DataFrame([{"plant": r_["plant"], "groups": r_["n_groups"], "effect d": r_["effect"], "oracle AUC gain": r_["oracle_auc_gain"],
                     "found": r_["detected"], "programs": r_["evaluated"]} for r_ in rows])
print(tab.sort_values(["plant", "effect d"]).to_string(index=False))

## 8 · What this does and does not show

* **Does:** inside its search space (integer transforms of single columns and their joint keys, read out through
  group means) there is no structure left in the residual of an honest blend that could be confirmed on unseen rows.
  The power table bounds how small a missed structure would have to be.
* **Does not:** exclude structures outside that space. Candidates worth testing, each with an acceptance criterion,
  are listed in the repository's [ROADMAP](https://github.com/Agnuxo1/s6e9-honest-ceiling/blob/main/ROADMAP.md): partial-pooling encodings of the income
  identity, non-mean readouts, relations between rows, learned or LLM-guided program proposals, and modelling the
  synthetic generator itself.
* **Why a negative result is worth publishing:** 47,679 programs is more feature engineering than any person tries by
  hand. Knowing that none of it survives a held-out fold saves everyone that time, and the harness is reusable on any
  tabular competition: replace the residual and run it.

If you take one of the open directions and beat these numbers honestly, open an issue in the repository: it will be
added with credit.